In [67]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal
from tqdm import tqdm

In [68]:
# Create Model
m = gp.Model("Model_1")

In [69]:
# GENERAL PARAMETERS

# Number of vehicle types
V = 2

# Number of nodes i,j
nodes = 4

# I, J = nodes, nodes
arcs = {i: [j for j in range(nodes) if abs(i - j) <= 1] for i in range(nodes)}

# Time Steps 12 (days) #testing with +1 day
T = 12

# Commodity variable types
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.INTEGER]
# Crew, consumables, equipment, samples, propellant, crew return

Y = GRB.INTEGER
# Spacecrafts of same type

In [70]:
# ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.0
water_consumption = 5.0
oxygen_consumption = 1.1
consumption = food_consumption + water_consumption + oxygen_consumption

# Crew mass [kg/crew]
crew_mass = 100

# Gravitational acceleration [m/sˆ2]
g_0 = 9.80665


In [71]:
# VEHICLE DATA

# Structure mass [kg]
s = np.array([40000, 15000, ])

# Specific impulses [s]
I_sp = np.array([421, 324])

# Payload Capacity [kg]
C = np.array([5000, 2500])

# Propellant Capacity [kg]
M = np.array([1200770, 400000])


In [72]:
# DISPLACEMENT DATA

# Velocity change [km/s]
# PAC, LEO, LLO, LS are 0, 1, 2, 3
delta_V = {0: {0: 0, 1: 0}, 1: {0: 0, 1: 0, 2: 4.04}, 2: {1: 4.04, 2: 0, 3: 1.87}, 3: {2: 1.87, 3: 0}}

# Time of travel [days]
TOF = {0: {0: 1, 1: 1}, 1: {0: 1, 1: 1, 2: 3}, 2: {1: 3, 2: 1, 3: 1}, 3: {2: 1, 3: 1}}

# Propellant mass fraction
phi = [[{j: 1 - np.exp(-(delta_V[i][j] / (I_sp[v] * g_0))) if I_sp[v] != 0 else 1 for j in arcs[i]}
        for i in arcs]
       for v in range(V)]


In [73]:
# CREATE COMMODITY FLOW VECTORS AND S/C COMMODITY FLOW
# Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'

def create_commodity_flow(model, V, arcs, T, X, direction):
    x_flow = [[{j: [np.array([[model.addVar(vtype=X[x], name=f'commodity_{direction}flow_{v},{i},{j},{t},{x}')]
                              for x in range(len(X))])
                    for t in range(T-1)]
                for j in arcs[i]}
               for i in arcs]
              for v in range(V)]

    return x_flow


def create_sc_commodity_flow(model, V, arcs, T, Y, direction):
    y_flow = [
        [{j: [np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},{t}')]) for t in range(T-1)]
          for j in arcs[i]}
         for i in arcs]
        for v in range(V)]

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, arcs, T, X, "out"), create_commodity_flow(m, V, arcs, T, X, "in")
y_outflow, y_inflow = create_sc_commodity_flow(m, V, arcs, T, Y, "out"), create_sc_commodity_flow(m, V, arcs, T, Y, "in")

m.update()


In [74]:
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]
# x = Crew, consumables, equipment, samples, propellant, crew return

# COMMODITY DEMAND

# THE COMMODITIES ARE PROVIDED AT LEO ???? START ALL THINGS AT LEO
# CHECK DAYS FOR MISSION !!!
#Commodity Array: D[Node][Day][Commodity]

D = [[np.array([0 for x in range(len(X))])
      for _ in range(T)]
    for _ in arcs]

print(len(D))
print(len(D[0]))
print(len(D[0][0]))

# Earth (PAC) crew, consumables, equipment, propellant supply infinite at leo time 0, AND Moon surface sample supply (infinite at all times)
D[1][0][0] = 99999 #Creww
D[1][0][1] = 99999 #Consumables
D[1][0][2] = 99999 #Equipment
D[1][0][4] = 9999999 #Propellant

for x in range(T):
    D[3][x][3] = 999999


# APOLLO

# for t in range(T):
#     D[1][t] = np.array([999999 if x != 3 else 0 for x in range(len(X))])
#     D[3][t][3] = 999999

# Crew demand/supply
D[3][4][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][3][0] = -1 # Lunar orbit day 4 crew demand
D[3][5][5] = 2 # Lunar surface day 6 crew supply (return)
D[2][6][5] = 1 # Lunar orbit day 7 crew supply (return)
D[0][10][5] = -3 # Earth day 11 crew demand (return)

D[3][4][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][10][3] = -110 # Earth day 11 lunar sample demand


"""
#attempting to move all demand+supply by 1 day
D[3][5][0] = -2 # Lunar surface day 6 crew demand (negative supply)
D[2][4][0] = -1 # Lunar orbit day 5 crew demand
D[3][6][5] = 2 # Lunar surface day 7 crew supply (return)
D[2][7][5] = 1 # Lunar orbit day 8 crew supply (return)
D[0][11][5] = -3 # Earth day 12 crew demand (return)

D[3][5][2] = -420 # Lunar surface day 6 (scientific) equipment demand

D[0][11][3] = -110 # Earth day 12 lunar sample demand
"""

# S/C COMMODITY DEMAND
# format d[node][vehicle][day]
d = [[[9999 if (i == 1 and t == 0) else 0 for t in range(T)] # Infinite supply of spacecrafts at i = 1, t = 0 LEO
     for _ in range(V)]
    for i in arcs]


4
12
6


In [75]:
# ADD THE CONSTRAINTS (2 & 3)

for i in arcs:
    for t in range(T):

        x_outflow_sum = sum(x_outflow[v][i][j][t] for v in range(V) for j in arcs[i]) if t < T-1 \
            else np.array([[0] for _ in range(len(X))])
        # On the last day there is no outflow
    

        x_inflow_sum = sum(x_inflow[v][j][i][t - TOF[j][i]] if t >= TOF[j][i]
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) for j in arcs[i])
        # Only count the inflows for which the spacecraft has had time to arrive

        for x in range(len(X)):
            m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x])

        # S/C commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t] for j in arcs[i]) if t < T-1 \
                else np.array([0])

            y_inflow_sum = sum(y_inflow[v][j][i][t - TOF[j][i]] if t >= TOF[j][i]
                               else np.array([0])
                               for j in arcs[i])

            m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t])


m.update()

In [76]:
# CONSTRAINTS 4 COMMODITY TRANSFORMATION

# Commodity transformation matrix
# Q[x+, y+] = [x-, y-] --> Difference between what leaves from node i and what arrives at node j. Ex. propellant use
# x = Crew, consumables, equipment, samples, propellant, crew return, then structure mass

def create_commodity_transformation(V, arcs, consumption, crew_mass, phi, TOF):
    Q = [[{j: np.array([[1, 0, 0, 0, 0, 0, 0],
                        [-consumption * TOF[i][j], 1, 0, 0, 0, -consumption * TOF[i][j], 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0, 0],
                        [0, 0, 0, 1, 0, 0, 0],
                        [crew_mass * -phi[v][i][j], -phi[v][i][j], -phi[v][i][j], -phi[v][i][j], 1 - phi[v][i][j], crew_mass * -phi[v][i][j], -phi[v][i][j]], # Propellant consumption
                        [0, 0, 0, 0, 0, 1, 0],
                        [0, 0, 0, 0, 0, 0, 1]])
           for j in arcs[i]}
          for i in arcs]
         for v in range(V)]

    return Q

Q = create_commodity_transformation(V, arcs, consumption, crew_mass, phi, TOF)

In [77]:
# ADD THE CONSTRAINTS (4)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for leaving, arriving in zip(np.dot(Q[v][i][j], np.concatenate((x_outflow[v][i][j][t],
                                                                                np.array([s[v]*y_outflow[v][i][j][t]])), axis=0)),
                                             np.concatenate((x_inflow[v][i][j][t], np.array([s[v]*y_inflow[v][i][j][t]])), axis=0)):

                    m.addConstr(leaving[0] == arriving[0])


# m.addConstr(Q[v][i][j] * np.concatenate((x_outflow[v][i][j][t], np.array([s[v]*y_outflow[v][i][j][t]])), axis=0) ==
#                             np.concatenate((x_inflow[v][i][j][t], np.array([s[v]*y_inflow[v][i][j][t]])), axis=0))

m.update()


In [78]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in s/c does not exceed maximum capacities
# x = Crew, consumables, equipment, samples, propellant, return crew

# def create_concurrency_constraint(V, arcs, crew_mass): # Different vehicles version
#     H = [[{j: np.array([[crew_mass, 1, 1, 1, 0],
#                         [0, 0, 0, 0, 1]])
#            for j in arcs[i]}
#           for i in arcs]
#          for v in range(V)]
#
#     return H


def create_concurrency_constraint(arcs, crew_mass): # Same for all vehicles, max payload mass
    H = [{j: np.array([[crew_mass, 1, 1, 1, 0, crew_mass],
                        [0, 0, 0, 0, 1, 0]])
           for j in arcs[i]}
          for i in arcs]
    return H


def create_sc_design_parameters(V, C, M):
    e = [np.array([[C[v]], [M[v]]]) for v in range(V)]
    return e


H = create_concurrency_constraint(arcs, crew_mass)
e = create_sc_design_parameters(V, C, M)
print(H)
print(len(H))

[{0: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 1: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]])}, {0: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 1: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 2: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]])}, {1: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 2: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 3: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]])}, {2: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]]), 3: array([[100,   1,   1,   1,   0, 100],
       [  0,   0,   0,   0,   1,   0]])}]
4


In [79]:
# ADD THE CONSTRAINTS (5)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for commodity, constraint in zip(np.dot(H[i][j], x_outflow[v][i][j][t]),
                                                 e[v]*y_outflow[v][i][j][t][0]):

                    m.addConstr(commodity[0] <= constraint[0])

m.update()


In [80]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

for v in range(V):
    for i in arcs:
        for j in arcs[i]:
            for t in range(T-1):
                for commodity_out in x_outflow[v][i][j][t]:
                    m.addConstr(commodity_out[0] >= 0)

                for commodity_in in x_inflow[v][i][j][t]:
                    m.addConstr(commodity_in[0] >= 0)

                m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# s[v] >= 0


In [81]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [82]:
# COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * s * y)

# x = Crew, consumables, equipment, samples, propellant, crew(return)

#Currently the model assumes we are starting at teh LEO node t = 0, i=0

def create_commodity_cost(V, arcs, crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1],[crew_mass]]) if (t == 0 and i == 1)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (t == 0 and i == 1)
                         else 0
                         for t in range(T-1)]
           for j in arcs[i]}
          for i in arcs]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff

cost_coeff, sc_cost_coeff = create_commodity_cost(V, arcs, crew_mass)

In [83]:
# DEFINE THE COST FUNCTION (1)

"""
#General version
cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * s[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in arcs
    for j in arcs[i]
    for t in range(3)
)
"""
#specific to apollo version
cost = sum(
    np.dot(cost_coeff[v][1][j][0].T, x_outflow[v][1][j][0]) + sc_cost_coeff[v][1][j][0] * s[v] * y_outflow[v][1][j][0][0]
    for v in range(V)
    for j in arcs[1]
)


cost = cost[0][0]
print(cost)

m.setObjective(cost, GRB.MINIMIZE)
m.update()

100.0 commodity_outflow_0,1,0,0,0 + commodity_outflow_0,1,0,0,1 + commodity_outflow_0,1,0,0,2 + commodity_outflow_0,1,0,0,3 + commodity_outflow_0,1,0,0,4 + 100.0 commodity_outflow_0,1,0,0,5 + 40000.0 sc_commodity_outflow_0,1,0,0 + 100.0 commodity_outflow_0,1,1,0,0 + commodity_outflow_0,1,1,0,1 + commodity_outflow_0,1,1,0,2 + commodity_outflow_0,1,1,0,3 + commodity_outflow_0,1,1,0,4 + 100.0 commodity_outflow_0,1,1,0,5 + 40000.0 sc_commodity_outflow_0,1,1,0 + 100.0 commodity_outflow_0,1,2,0,0 + commodity_outflow_0,1,2,0,1 + commodity_outflow_0,1,2,0,2 + commodity_outflow_0,1,2,0,3 + commodity_outflow_0,1,2,0,4 + 100.0 commodity_outflow_0,1,2,0,5 + 40000.0 sc_commodity_outflow_0,1,2,0 + 100.0 commodity_outflow_1,1,0,0,0 + commodity_outflow_1,1,0,0,1 + commodity_outflow_1,1,0,0,2 + commodity_outflow_1,1,0,0,3 + commodity_outflow_1,1,0,0,4 + 100.0 commodity_outflow_1,1,0,0,5 + 15000.0 sc_commodity_outflow_1,1,0,0 + 100.0 commodity_outflow_1,1,1,0,0 + commodity_outflow_1,1,1,0,1 + commodity_

In [84]:
m.optimize()

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i9-10885H CPU @ 2.40GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 5444 rows, 3080 columns and 11912 nonzeros
Model fingerprint: 0x9446d2df
Variable types: 1760 continuous, 1320 integer (0 binary)
Coefficient statistics:
  Matrix range     [5e-04, 1e+06]
  Objective range  [1e+00, 4e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+07]
Presolve removed 5136 rows and 2683 columns
Presolve time: 0.01s
Presolved: 308 rows, 397 columns, 1384 nonzeros
Variable types: 250 continuous, 147 integer (2 binary)

Root relaxation: objective 1.591777e+04, 351 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 15917.7658    0   

In [ ]:
# x = Crew, consumables, equipment, samples, propellant
#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("apollo_solution.txt", "w")




for final in sorted_results:
    if sorted_results[final] != 0:
        print('%s %g' % (final, sorted_results[final]))
        f.write('%s %g' % (final, sorted_results[final]))
        f.write('\n')
f.close()


commodity_outflow_1,1,2,0,0 3
commodity_outflow_1,1,2,0,1 177.5
commodity_outflow_1,1,2,0,2 420
commodity_outflow_1,1,2,0,4 58.3524
commodity_inflow_1,1,2,0,0 3
commodity_inflow_1,1,2,0,1 113.6
commodity_inflow_1,1,2,0,2 420
commodity_inflow_1,1,2,0,4 38.0775
sc_commodity_outflow_1,1,2,0 1
sc_commodity_inflow_1,1,2,0 1
commodity_outflow_1,2,3,3,0 2
commodity_outflow_1,2,3,3,1 113.6
commodity_outflow_1,2,3,3,2 420
commodity_outflow_1,2,3,3,4 38.0775
commodity_inflow_1,2,3,3,0 2
commodity_inflow_1,2,3,3,1 99.4
commodity_inflow_1,2,3,3,2 420
commodity_inflow_1,2,3,3,4 28.798
sc_commodity_outflow_1,2,3,3 1
sc_commodity_inflow_1,2,3,3 1
commodity_outflow_1,3,3,4,1 99.4
commodity_outflow_1,3,3,4,4 28.798
commodity_inflow_1,3,3,4,1 99.4
commodity_inflow_1,3,3,4,4 28.798
sc_commodity_outflow_1,3,3,4 1
sc_commodity_inflow_1,3,3,4 1
commodity_outflow_1,3,2,5,1 99.4
commodity_outflow_1,3,2,5,3 110
commodity_outflow_1,3,2,5,4 28.798
commodity_outflow_1,3,2,5,5 2
commodity_inflow_1,3,2,5,1 85.2
com

In [ ]:
# Making a graph
keylist =list(sorted_results.keys())
SC_Outflow =[x for x in keylist if 'sc_commodity_outflow' in x]

In [86]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # s
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [87]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )